# HYT ses klonu — Chatterbox Multilingual (MIT lisans)
Runtime > Change runtime type > **T4 GPU**. Hücreleri sırayla çalıştır.

- Model: Resemble AI Chatterbox Multilingual. MIT lisanslı, Türkçe destekli, ticari kullanıma uygun.
- Eski defterdeki XTTS-v2 CPML lisanslı; ticari kullanım (Instagram lead, reklam) için uygun değil.
- Referans: `video-klon/ref/hasan_ref_kisa_*.wav` (yalnızca kendi sesin, rıza: Hasan Yasin Türkyılmaz).
- Girdi: `video-klon/senaryo/*.json` · Çıktı: `video-klon/cikti/<id>/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/Claude/Projects/hyt growth/video-klon'
import os; print(os.listdir(BASE+'/ref')); print(os.listdir(BASE+'/senaryo'))


In [ ]:
!pip -q install chatterbox-tts


In [ ]:
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_pretrained(device='cuda')
SR = model.sr


## 1) En iyi referansı seç
Her referansla aynı cümleyi üretir. Dinle, en doğal olanın numarasını `REF_NO` olarak yaz.


In [ ]:
from IPython.display import Audio, display
TEST='Merhaba, ben Hasan Yasin Türkyılmaz. Bugün yapay zekâ ajanlarının e-ticarete etkisini konuşuyoruz.'
os.makedirs(BASE+'/cikti/_test', exist_ok=True)
for i in range(1,7):
    ref=f'{BASE}/ref/hasan_ref_kisa_{i}.wav'
    w=model.generate(TEST, language_id='tr', audio_prompt_path=ref, exaggeration=0.5, cfg_weight=0.5)
    p=f'{BASE}/cikti/_test/ref{i}.wav'; ta.save(p, w, SR); print('ref', i); display(Audio(p))


## 2) Senaryoyu seslendir
`exaggeration` vurgu (0.3 sakin, 0.7 enerjik). `cfg_weight` düşükse daha yavaş, tok konuşur.
Her bölüm ayrı WAV + birleşik `tam.wav` + `zamanlama.json` üretir. Montaj aracı bu dosyaları kullanır.


In [ ]:
import json, glob
REF_NO = 1
EXAG, CFG = 0.5, 0.4
BOSLUK = 0.30   # bölümler arası sessizlik (sn)
ref=f'{BASE}/ref/hasan_ref_kisa_{REF_NO}.wav'
for yol in sorted(glob.glob(BASE+'/senaryo/*.json')):
    s=json.load(open(yol, encoding='utf-8'))
    out=f"{BASE}/cikti/{s['id']}"; os.makedirs(out, exist_ok=True)
    parcalar, zaman, t = [], [], 0.0
    for b in s['bolumler']:
        w=model.generate(b['tts'], language_id='tr', audio_prompt_path=ref, exaggeration=EXAG, cfg_weight=CFG)
        ta.save(f"{out}/{b['id']}.wav", w, SR)
        sure=w.shape[-1]/SR
        zaman.append({'id':b['id'], 'slayt':b['slayt'], 'bas':round(t,3), 'sure':round(sure,3), 'altyazi':b['altyazi']})
        parcalar += [w, torch.zeros(1, int(SR*BOSLUK))]; t += sure + BOSLUK
    ta.save(f'{out}/tam.wav', torch.cat(parcalar, dim=-1), SR)
    json.dump({'id':s['id'],'sr':SR,'bolumler':zaman}, open(f'{out}/zamanlama.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)
    print('tamam', s['id'], round(t,1), 'sn'); display(Audio(f'{out}/tam.wav'))


## 3) Sonraki adım
`cikti/<id>/tam.wav` ve `zamanlama.json` dosyalarını Claude'a yükle (sohbete sürükle-bırak).
Montaj: `python3 icerik/araclar/build-reels.py` slaytları sese göre zamanlar, %9 müzik ekler, altyazıyı basar.
